# Dota 2 Daily Data Collection Job

Collecting match data from the OpenDota API and saving it to my Delta tables.

**Schedule:** Runs every 90 mins

Content:
- Live/recent matches from `/live` endpoint
- Detailed match data from `/matches/{match_id}` (sample)
- Stores in Delta tables: `dota2_live_matches`, `dota2_detailed_matches`

Future scope: Build historical dataset for ML model training and analytics

## API Quota Optimization

**Daily quota:** 3,000 requests/day

**Current usage breakdown:**
- Live matches: **1 request** (lightweight, collects match metadata)
- Pro matches: **1,200 requests** (completed, high-skill matches)
- Public matches: **1,200 requests** (variety of skill levels)
- Hero item popularity: **30 requests** (item usage by phase for top 30 heroes)

**Total:** ~2,431 requests/day (**81% of quota**)

**What we collect:**

| Data Type | API Calls | Records/Day | Match Quality | Purpose |
|-----------|-----------|-------------|---------------|----------|
| Live matches | 1 | ~50-100 | Real-time snapshot | Match metadata |
| Pro matches | 1,200 | 1,200 | ✅ Completed (30-75 min) | High-skill ML training |
| Public matches | 1,200 | 1,200 | ⚠️ Mixed (some incomplete) | Skill diversity, all ranks |
| Item popularity | 30 | ~3,000-5,000 | Top 30 heroes | Item usage patterns |

**Collection benefits:**
- **Pro matches**: Guaranteed complete games, high skill (Divine+), Captain's Mode
- **Public matches**: All skill brackets (Herald → Immortal), multiple game modes
- **Automatic flags**: `match_type` (pro/public) + `is_turbo` (true/false) for easy filtering

**After 30 days:** ~36,000 pro + ~36,000 public matches + ~120,000 item popularity records

**Future expansion options:**
- Maximize to 2,969 detailed matches/day (use full 99.7% quota)
- Add hero matchup win rates (requires additional API calls)
- Add player-specific data (requires additional API calls)
- Add patch-specific tracking

In [0]:
import requests
import time
import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

BASE_URL = "https://api.opendota.com/api"
HEADERS = {"Accept": "application/json"}

print(f"collection date: {datetime.date.today().isoformat()}")

collection date: 2026-08-28


In [0]:
%sql

SELECT 
    collection_date,
    collection_timestamp,
    COUNT(*) as matches_collected,
    TIMESTAMPDIFF(MINUTE, LAG(collection_timestamp) OVER (ORDER BY collection_timestamp), collection_timestamp) as minutes_since_last_run
FROM dota2_detailed_matches
-- WHERE collection_date = CURRENT_DATE()
GROUP BY collection_date, collection_timestamp
ORDER BY collection_timestamp DESC;

collection_date,collection_timestamp,matches_collected,minutes_since_last_run
2026-08-28,2026-08-28T08:43:16.909Z,100,1
2026-08-28,2026-08-28T08:41:23.115Z,100,2
2026-08-28,2026-08-28T08:38:47.109Z,5,0
2026-08-28,2026-08-28T08:38:35.956Z,10,0
2026-08-28,2026-08-28T08:38:22.609Z,10,0
2026-08-28,2026-08-28T08:38:05.161Z,10,84
2026-08-28,2026-08-28T07:13:33.304Z,100,1
2026-08-28,2026-08-28T07:11:37.617Z,98,2
2026-08-28,2026-08-28T07:08:55.986Z,5,0
2026-08-28,2026-08-28T07:08:44.763Z,10,0


In [0]:
dota2_detailed_matches = spark.table("dota2_detailed_matches")

display(dota2_detailed_matches)

duration game_mode match_id patch radiant_win raw_json region start_time collection_date collection_timestamp 0 1 8958565057 60 false {"players": [{"account_id": 1799887380, "player_slot": 0, "team_number": 0, "team_slot": 0, "hero_id": 0, "hero_variant": 0, "item_0": 0, "item_1": 0, "item_2": 0, "item_3": 0, "item_4": 0, "item_5": 0, "backpack_0": 0, "backpack_1": 0, "backpack_2": 0, "item_neutral": 0, "item_neutral2": 0, "kills": 0, "deaths": 0, "assists": 0, "leaver_status": 1, "last_hits": 0, "denies": 0, "gold_per_min": 0, "xp_per_min": 0, "level": 0, "net_worth": 0, "aghanims_scepter": 0, "aghanims_shard": 0, "moonshard": 0, "hero_damage": 0, "tower_damage": 0, "hero_healing": 0, "gold": 0, "gold_spent": 0, "personaname": "inactive", "name": null, "last_login": "2025-03-08T10:59:37.496Z", "rank_tier": 53, "computed_mmr": 3995.14, "is_subscriber": false, "radiant_win": false, "start_time": 1787371542, "duration": 0, "cluster": 181, "lobby_type": 4, "game_mode": 1, "is_contributor": false, "patch": 60, "region": 8, "isRadiant": true, "win": 0, "lose": 1, "kda": 0, "abandons": 0, "benchmarks": {"gold_per_min": {"raw": 0, "pct": null}, "xp_per_min": {"raw": 0, "pct": null}, "kills_per_min": {"raw": null}, "deaths_per_min": {"raw": null}, "assists_per_min": {"raw": null}, "last_hits_per_min": {"raw": null}, "denies_per_min": {"raw": null}, "hero_damage_per_min": {"raw": null}, "hero_healing_per_min": {"raw": null}, "tower_damage": {"raw": 0, "pct": null}}}, {"player_slot": 1, "team_number": 0, "team_slot": 1, "hero_id": 0, "hero_variant": 0, "item_0": 0, "item_1": 0, "item_2": 0, "item_3": 0, "item_4": 0, "item_5": 0, "backpack_0": 0, "backpack_1": 0, "backpack_2": 0, "item_neutral": 0, "item_neutral2": 0, "kills": 0, "deaths": 0, "assists": 0, "hero_damage": 0, "tower_damage": 0, "hero_healing": 0, "gold": 0, "last_hits": 0, "denies": 0, "gold_per_min": 7860, "xp_per_min": 0, "gold_spent": 0, "level": 0, "net_worth": 0, "is_subscriber": false, "radiant_win": false, "start_time": 1787371542, "duration": 0, "cluster": 181, "lobby_type": 4, "game_mode": 1, "is_contributor": false, "patch": 60, "region": 8, "isRadiant": true, "win": 0, "lose": 1, "kda": 0, "benchmarks": {"gold_per_min": {"raw": 7860, "pct": null}, "xp_per_min": {"raw": 0, "pct": null}, "kills_per_min": {"raw": null}, "deaths_per_min": {"raw": null}, "assists_per_min": {"raw": null}, "last_hits_per_min": {"raw": null}, "denies_per_min": {"raw": null}, "hero_damage_per_min": {"raw": null}, "hero_healing_per_min": {"raw": null}, "tower_damage": {"raw": 0, "pct": null}}}, {"player_slot": 2, "team_number": 0, "team_slot": 2, "hero_id": 0, "hero_variant": 0, "item_0": 0, "item_1": 0, "item_2": 0, "item_3": 0, "item_4": 0, "item_5": 0, "backpack_0": 0, "backpack_1": 0, "backpack_2": 0, "item_neutral": 0, "item_neutral2": 0, "kills": 0, "deaths": 0, "assists": 0, "hero_damage": 0, "tower_damage": 0, "hero_healing": 0, "gold": 0, "last_hits": 0, "denies": 0, "gold_per_min": 7860, "xp_per_min": 0, "gold_spent": 0, "level": 0, "net_worth": 0, "is_subscriber": false, "radiant_win": false, "start_time": 1787371542, "duration": 0, "cluster": 181, "lobby_type": 4, "game_mode": 1, "is_contributor": false, "patch": 60, "region": 8, "isRadiant": true, "win": 0, "lose": 1, "kda": 0, "benchmarks": {"gold_per_min": {"raw": 7860, "pct": null}, "xp_per_min": {"raw": 0, "pct": null}, "kills_per_min": {"raw": null}, "deaths_per_min": {"raw": null}, "assists_per_min": {"raw": null}, "last_hits_per_min": {"raw": null}, "denies_per_min": {"raw": null}, "hero_damage_per_min": {"raw": null}, "hero_healing_per_min": {"raw": null}, "tower_damage": {"raw": 0, "pct": null}}}, {"player_slot": 3, "team_number": 0, "team_slot": 3, "hero_id": 0, "hero_variant": 0, "item_0": 0, "item_1": 0, "item_2": 0, "item_3": 0, "item_4": 0, "item_5": 0, "backpack_0": 0, "backpack_1": 0, "backpack_2": 0, "item_neutral": 0, "item_neutral2": 0, "kills": 0, "deaths": 0, "assists": 0, "hero_damage": 0, "tower_damage"

In [0]:
def collect_live_matches():
    today = datetime.date.today().isoformat()
    
    live_url = f"{BASE_URL}/live"
    print(f"Fetching live matches from {live_url}...")
    
    response = requests.get(live_url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"Failed to fetch live matches: {response.status_code}")
        return None
    
    matches = response.json()
    print(f"Collected {len(matches)} live matches")
    
    if not matches:
        print("No live matches found")
        return None

    import json
    
    processed_matches = []
    for match in matches:
        processed_matches.append({
            'match_id': match.get('match_id'),
            'server_steam_id': match.get('server_steam_id'),
            'lobby_type': match.get('lobby_type'),
            'game_mode': match.get('game_mode'),
            'average_mmr': match.get('average_mmr'),
            'game_time': match.get('game_time'),
            'spectators': match.get('spectators'),
            'raw_json': json.dumps(match)  
        })
    
    matches_df = spark.createDataFrame(processed_matches)
    matches_df = matches_df.withColumn("collection_date", F.lit(today))
    matches_df = matches_df.withColumn("collection_timestamp", F.current_timestamp())
    
    matches_df.write.format("delta") \
        .mode("append") \
        .saveAsTable("dota2_live_matches")
    
    print(f"Saved {matches_df.count()} matches to table: dota2_live_matches")
    
    return matches_df

In [0]:
def collect_detailed_matches(sample_size=50, use_pro_matches=True, match_type_label=None):
    if use_pro_matches:
        matches_url = f"{BASE_URL}/proMatches"
        match_type = match_type_label or "pro"
        print(f"Fetching completed pro matches from {matches_url}")
    else:
        matches_url = f"{BASE_URL}/publicMatches"
        match_type = match_type_label or "public"
        print(f"Fetching public matches from {matches_url} (may include in-progress)")
    
    response = requests.get(matches_url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"Failed to fetch matches: {response.status_code}")
        return None
    
    matches = response.json()
    print(f"Found {len(matches)} matches")
    
    match_ids = [m['match_id'] for m in matches[:sample_size]]
    print(f"Found {len(match_ids)} recent match IDs")
    
    detailed_matches = []
    failed_count = 0
    
    for idx, match_id in enumerate(match_ids, 1):
        url = f"{BASE_URL}/matches/{match_id}"
        response = requests.get(url, headers=HEADERS)
        
        if response.status_code == 200:
            match_data = response.json()
            detailed_matches.append(match_data)
            if idx % 10 == 0:
                print(f"  Progress: {idx}/{len(match_ids)} matches collected")
        else:
            failed_count += 1
            if failed_count <= 3:
                print(f"Failed to fetch match {match_id}: {response.status_code}")
        
        time.sleep(1)
    
    print(f"Collected {len(detailed_matches)} detailed matches")
    if failed_count > 0:
        print(f"{failed_count} matches failed to fetch")
    
    if not detailed_matches:
        print("No detailed match data collected")
        return None
    
    import json
    
    schema = StructType([
        StructField("match_id", LongType(), True),
        StructField("duration", IntegerType(), True),
        StructField("game_mode", IntegerType(), True),
        StructField("is_turbo", BooleanType(), True),
        StructField("is_likely_complete", BooleanType(), True),
        StructField("radiant_win", BooleanType(), True),
        StructField("start_time", IntegerType(), True),
        StructField("patch", IntegerType(), True),
        StructField("region", IntegerType(), True),
        StructField("match_type", StringType(), True),
        StructField("raw_json", StringType(), True)
    ])
    
    processed_matches = []
    for match in detailed_matches:
        game_mode = match.get('game_mode')
        duration = match.get('duration', 0)
        is_turbo = game_mode == 23

        if match_type == "pro":
            is_likely_complete = True
        elif is_turbo:
            is_likely_complete = duration >= 600
        else:
            is_likely_complete = duration >= 1500
        
        processed_matches.append((
            match.get('match_id'),
            duration,
            game_mode,
            is_turbo,
            is_likely_complete,
            match.get('radiant_win'),
            match.get('start_time'),
            match.get('patch'),
            match.get('region'),
            match_type,
            json.dumps(match)
        ))
    
    matches_df = spark.createDataFrame(processed_matches, schema=schema)
    matches_df = matches_df.withColumn("collection_date", F.lit(datetime.date.today().isoformat()))
    matches_df = matches_df.withColumn("collection_timestamp", F.current_timestamp())
    
    matches_df.write.format("delta") \
        .mode("append") \
        .saveAsTable("dota2_detailed_matches")
    
    print(f"Saved {len(detailed_matches)} detailed matches to table: dota2_detailed_matches")
    
    return matches_df

In [0]:
table_desc = spark.sql("DESCRIBE dota2_detailed_matches").collect()
column_names = [row.col_name for row in table_desc]

if 'is_likely_complete' not in column_names:
    spark.sql("""
        ALTER TABLE dota2_detailed_matches 
        ADD COLUMN is_likely_complete BOOLEAN AFTER is_turbo
    """)
    print("Column added successfully")
else:
    print("Column is_likely_complete already exists, skipping this")

spark.sql("""
    UPDATE dota2_detailed_matches 
    SET is_likely_complete = CASE 
        WHEN match_type = 'pro' THEN true
        WHEN is_turbo = true AND duration >= 600 THEN true
        WHEN is_turbo = false AND duration >= 1500 THEN true
        ELSE false
    END
    WHERE is_likely_complete IS NULL
""")
print("Update complete")

display(spark.sql("DESCRIBE dota2_detailed_matches"))

col_name,data_type,comment
duration,bigint,null
game_mode,bigint,null
is_turbo,boolean,null
is_likely_complete,boolean,null
match_id,bigint,null
patch,bigint,null
radiant_win,boolean,null
raw_json,string,null
region,bigint,null
match_type,string,null


In [0]:
%sql
UPDATE dota2_detailed_matches 
SET is_likely_complete = CASE 
    WHEN match_type = 'pro' THEN true
    WHEN is_turbo = true AND duration >= 600 THEN true
    WHEN is_turbo = false AND duration >= 1500 THEN true
    ELSE false
END
WHERE is_likely_complete IS NULL;

SELECT 
    match_type,
    is_turbo,
    is_likely_complete,
    COUNT(*) as matches,
    ROUND(AVG(duration) / 60, 1) as avg_duration_min,
    MIN(duration) as min_duration,
    MAX(duration) as max_duration
FROM dota2_detailed_matches
GROUP BY match_type, is_turbo, is_likely_complete
ORDER BY match_type, is_turbo, is_likely_complete;

match_type,is_turbo,is_likely_complete,matches,avg_duration_min,min_duration,max_duration
pro,false,true,10,49.2,2171,4493
public,false,false,441,7.9,0,1352
public,false,true,10,49.2,2171,4493
public,true,false,276,7.1,374,588
public,true,true,573,16.3,605,1395


In [0]:
test_df = collect_detailed_matches(sample_size=10, use_pro_matches=True)

if test_df:
    test_df.select("match_id", "duration", "game_mode").show(10, False)
    
    avg_duration = test_df.agg({"duration": "avg"}).collect()[0][0]
    min_duration = test_df.agg({"duration": "min"}).collect()[0][0]
    max_duration = test_df.agg({"duration": "max"}).collect()[0][0]
    
    print(f"\nAverage duration: {avg_duration:.0f} sec ({avg_duration/60:.1f} min)")
    print(f"Min duration: {min_duration} sec ({min_duration/60:.1f} min)")
    print(f"Max duration: {max_duration} sec ({max_duration/60:.1f} min)")
    print(f"\nExpected range: 1800-3000 sec (30-50 min) for completed matches")

Testing pro matches collection...
Fetching completed pro matches from https://api.opendota.com/api/proMatches
Found 100 matches
Found 10 recent match IDs
  Progress: 10/10 matches collected
Collected 10 detailed matches
Saved 10 detailed matches to table: dota2_detailed_matches

=== DURATION VERIFICATION ===
+----------+--------+---------+
|match_id  |duration|game_mode|
+----------+--------+---------+
|8958842370|3163    |2        |
|8958734008|2610    |2        |
|8958667855|2171    |2        |
|8958607830|2338    |2        |
|8958538533|2601    |2        |
|8958478716|2974    |2        |
|8957890399|3672    |2        |
|8957749452|2688    |2        |
|8957527957|4493    |2        |
|8957364447|2813    |2        |
+----------+--------+---------+


Average duration: 2952 sec (49.2 min)
Min duration: 2171 sec (36.2 min)
Max duration: 4493 sec (74.9 min)

Expected range: 1800-3000 sec (30-50 min) for completed matches


In [0]:
pro_df = collect_detailed_matches(sample_size=10, use_pro_matches=True)

public_df = collect_detailed_matches(sample_size=10, use_pro_matches=False)

combined = spark.table("dota2_detailed_matches")

combined.groupBy("match_type").count().show()

print("\nDuration comparison by match type:")
display(combined.groupBy("match_type").agg(
    F.count("*").alias("matches"),
    F.round(F.avg("duration"), 0).alias("avg_duration_sec"),
    F.round(F.avg("duration") / 60, 1).alias("avg_duration_min"),
    F.min("duration").alias("min_duration"),
    F.max("duration").alias("max_duration")
).orderBy("match_type"))

=== COLLECTING PRO MATCHES ===
Fetching completed pro matches from https://api.opendota.com/api/proMatches
Found 100 matches
Found 10 recent match IDs
  Progress: 10/10 matches collected
Collected 10 detailed matches
Saved 10 detailed matches to table: dota2_detailed_matches

=== COLLECTING PUBLIC MATCHES ===
Fetching public matches from https://api.opendota.com/api/publicMatches (may include in-progress)
Found 100 matches
Found 10 recent match IDs
  Progress: 10/10 matches collected
Collected 10 detailed matches
Saved 10 detailed matches to table: dota2_detailed_matches

=== COMBINED RESULTS ===
Match type distribution:
+----------+-----+
|match_type|count|
+----------+-----+
|    public| 1295|
|       pro|   10|
+----------+-----+


Duration comparison by match type:


match_type,matches,avg_duration_sec,avg_duration_min,min_duration,max_duration
pro,10,2952.0,49.2,2171,4493
public,1295,706.0,11.8,0,4493


In [0]:
test_turbo = collect_detailed_matches(sample_size=5, use_pro_matches=False)

if test_turbo:
    display(test_turbo.select(
        "match_id",
        "game_mode",
        "is_turbo",
        "duration",
        F.round(F.col("duration") / 60, 1).alias("duration_min"),
        "match_type"
    ))
    
    print("\nBreakdown by Turbo status:")
    display(spark.table("dota2_detailed_matches").groupBy("is_turbo", "match_type").agg(
        F.count("*").alias("matches"),
        F.round(F.avg("duration") / 60, 1).alias("avg_duration_min")
    ).orderBy("match_type", "is_turbo"))

Testing is_turbo flag with public matches (includes Turbo)...
Fetching public matches from https://api.opendota.com/api/publicMatches (may include in-progress)
Found 100 matches
Found 5 recent match IDs
Collected 5 detailed matches
Saved 5 detailed matches to table: dota2_detailed_matches

=== TURBO FLAG VERIFICATION ===


match_id,game_mode,is_turbo,duration,duration_min,match_type
8959025703,13,false,0,0.0,public
8959024343,1,false,0,0.0,public
8959022633,13,false,0,0.0,public
8959021587,13,false,0,0.0,public
8959019860,1,false,0,0.0,public



Breakdown by Turbo status:


is_turbo,match_type,matches,avg_duration_min
false,pro,10,49.2
false,public,451,8.8
true,public,849,13.3


In [0]:
def collect_hero_item_popularity(hero_ids_sample=30):
    heroes_url = f"{BASE_URL}/heroStats"
    response = requests.get(heroes_url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"Failed to fetch hero stats: {response.status_code}")
        return None
    
    heroes_data = response.json()
    popular_heroes = sorted(heroes_data, key=lambda x: x.get('pro_pick', 0), reverse=True)[:hero_ids_sample]
    hero_ids = [h['id'] for h in popular_heroes]
    
    print(f"Collecting item popularity for {len(hero_ids)} most-picked heroes...")
    
    item_popularity_records = []
    failed_count = 0
    
    for idx, hero_id in enumerate(hero_ids, 1):
        url = f"{BASE_URL}/heroes/{hero_id}/itemPopularity"
        response = requests.get(url, headers=HEADERS)
        
        if response.status_code == 200:
            data = response.json()
            hero_name = next((h['localized_name'] for h in heroes_data if h['id'] == hero_id), f"Hero_{hero_id}")
            
            for phase in ['start_game_items', 'early_game_items', 'mid_game_items', 'late_game_items']:
                if phase in data and isinstance(data[phase], dict):
                    for item_id, count in data[phase].items():
                        item_popularity_records.append({
                            'hero_id': hero_id,
                            'hero_name': hero_name,
                            'item_id': item_id,
                            'phase': phase.replace('_items', ''),
                            'popularity_count': count
                        })
            
            if idx % 10 == 0:
                print(f"  Progress: {idx}/{len(hero_ids)} heroes")
        else:
            failed_count += 1
            if failed_count <= 3:
                print(f"  Failed hero_id {hero_id}: {response.status_code}")
        
        time.sleep(1)  
    
    
    if not item_popularity_records:
        print("No item popularity data collected")
        return None
    
    df = spark.createDataFrame(item_popularity_records)
    df = df.withColumn("collection_date", F.lit(datetime.date.today().isoformat()))
    df = df.withColumn("collection_timestamp", F.current_timestamp())
    
    df.write.format("delta") \
        .mode("append") \
        .saveAsTable("dota2_hero_item_popularity")
    
    print(f"Saved {len(item_popularity_records)} item popularity records to table: dota2_hero_item_popularity")
    
    
    return df

In [0]:
import datetime
print(f"Started at: {datetime.datetime.now().isoformat()}")

live_df = collect_live_matches()

print("COLLECTING PRO MATCHES (completed, high-skill)")
print("="*60)
pro_df = collect_detailed_matches(sample_size=1200, use_pro_matches=True)

print("COLLECTING PUBLIC MATCHES (variety of skill levels)")
print("="*60)
public_df = collect_detailed_matches(sample_size=1200, use_pro_matches=False)

item_pop_df = collect_hero_item_popularity(hero_ids_sample=30)

print(f"\nFinished at: {datetime.datetime.now().isoformat()}")

print("COLLECTION SUMMARY")

if live_df:
    print(f"Live matches: {live_df.count()}")
if pro_df:
    print(f"Pro matches: {pro_df.count()} (completed, 30-75 min durations)")
if public_df:
    print(f"Public matches: {public_df.count()} (mixed durations, all skill levels)")
if item_pop_df:
    print(f"Item popularity records: {item_pop_df.count()}")

api_calls = 1  # live matches
if pro_df:
    api_calls += 1200  # pro detailed matches
if public_df:
    api_calls += 1200  # public detailed matches
if item_pop_df:
    api_calls += 30  # item popularity

total_matches = (pro_df.count() if pro_df else 0) + (public_df.count() if public_df else 0)

print(f"\nTotal detailed matches: {total_matches}")
print(f"Total API calls: ~{api_calls}")
print(f"Daily quota: 3,000")
print(f"Quota used: {(api_calls/3000)*100:.1f}%")
print(f"Remaining: ~{3000-api_calls} calls")

Started at: 2026-08-22T11:12:17.526414
Fetching live matches from https://api.opendota.com/api/live...
Collected 100 live matches
Saved 100 matches to table: dota2_live_matches

COLLECTING PRO MATCHES (completed, high-skill)
Fetching completed pro matches from https://api.opendota.com/api/proMatches
Found 100 matches
Found 100 recent match IDs
  Progress: 10/100 matches collected
  Progress: 20/100 matches collected
  Progress: 30/100 matches collected
  Progress: 40/100 matches collected
  Progress: 50/100 matches collected
  Progress: 60/100 matches collected
  Progress: 70/100 matches collected
  Progress: 80/100 matches collected
  Progress: 90/100 matches collected
  Progress: 100/100 matches collected
Collected 100 detailed matches
Saved 100 detailed matches to table: dota2_detailed_matches

COLLECTING PUBLIC MATCHES (variety of skill levels)
Fetching public matches from https://api.opendota.com/api/publicMatches (may include in-progress)
Found 100 matches
Found 100 recent match 

In [0]:
%sql
SELECT 
    match_id,
    average_mmr,
    game_mode,
    lobby_type,
    spectators,
    game_time,
    collection_date,
    collection_timestamp
FROM dota2_live_matches
ORDER BY collection_timestamp DESC
LIMIT 10;

match_id,average_mmr,game_mode,lobby_type,spectators,game_time,collection_date,collection_timestamp
8956698237,6243,22,7,5,2124,2026-08-21,2026-08-21T03:01:01.818Z
8956695068,5751,22,7,2,2746,2026-08-21,2026-08-21T03:01:01.818Z
8956696996,7972,22,7,31,364,2026-08-21,2026-08-21T03:01:01.818Z
8956697227,5861,22,7,3,2330,2026-08-21,2026-08-21T03:01:01.818Z
8956690973,7465,22,7,0,958,2026-08-21,2026-08-21T03:01:01.818Z
8956694271,5738,22,7,0,-40,2026-08-21,2026-08-21T03:01:01.818Z
8956690668,7628,22,7,63,993,2026-08-21,2026-08-21T03:01:01.818Z
8956694414,7713,22,7,31,2248,2026-08-21,2026-08-21T03:01:01.818Z
8956692673,5775,22,7,1,929,2026-08-21,2026-08-21T03:01:01.818Z
8956699670,7944,22,7,33,1413,2026-08-21,2026-08-21T03:01:01.818Z


Game mode legend and meaning:
Code
Mode Name
Description
1
All Pick
Casual version - all heroes available
2
Captain's Mode
Competitive draft mode with bans
3
Random Draft
Pick from a pool of 50 random heroes
4
Single Draft
Each player picks from 3 random heroes
5
All Random
All players get random heroes
22
All Pick Ranked
Ranked matchmaking with all heroes available
23
Turbo
Faster-paced version with increased gold/XP


In [0]:
%sql
SELECT 
    hero_name,
    item_id,
    phase,
    popularity_count,
    collection_date
FROM dota2_hero_item_popularity
WHERE phase = 'start_game'
ORDER BY popularity_count DESC
LIMIT 20;

hero_name,item_id,phase,popularity_count,collection_date
Pangolier,16,start_game,428,2026-08-19
Earth Spirit,16,start_game,411,2026-08-19
Ember Spirit,16,start_game,392,2026-08-19
Puck,16,start_game,390,2026-08-19
Lina,16,start_game,316,2026-08-19
Dark Seer,16,start_game,280,2026-08-19
Slardar,16,start_game,265,2026-08-19
Timbersaw,16,start_game,224,2026-08-19
Invoker,16,start_game,216,2026-08-19
Dark Willow,16,start_game,211,2026-08-19


In [0]:
%sql
SELECT 
    collection_date,
    COUNT(DISTINCT match_id) as live_matches
FROM dota2_live_matches
GROUP BY collection_date
ORDER BY collection_date DESC;

SELECT 
    collection_date,
    COUNT(*) as detailed_matches,
    AVG(duration) as avg_duration_seconds,
    SUM(CASE WHEN radiant_win THEN 1 ELSE 0 END) as radiant_wins,
    SUM(CASE WHEN NOT radiant_win THEN 1 ELSE 0 END) as dire_wins
FROM dota2_detailed_matches
GROUP BY collection_date
ORDER BY collection_date DESC;

SELECT 
    collection_date,
    COUNT(*) as item_records,
    COUNT(DISTINCT hero_name) as heroes_tracked
FROM dota2_hero_item_popularity
GROUP BY collection_date
ORDER BY collection_date DESC;

collection_date,item_records,heroes_tracked
2026-08-19,2489,30


In [0]:
from pyspark.sql.functions import get_json_object, expr, col, when

dota2_detailed_matches = spark.table("dota2_detailed_matches")

df_ranked = dota2_detailed_matches.select(
    "match_id",
    "collection_date",
    "duration",
    "radiant_win",
    "game_mode",
    *[get_json_object("raw_json", f"$.players[{i}].rank_tier").cast("int").alias(f"player_{i}_rank") 
      for i in range(10)]
)

rank_cols = [f"player_{i}_rank" for i in range(10)]
sum_expr = ' + '.join([f'coalesce({col}, 0)' for col in rank_cols])
count_expr = ' + '.join([f'case when {col} is not null then 1 else 0 end' for col in rank_cols])

df_with_avg_rank = df_ranked.withColumn(
    "avg_rank",
    expr(f"({sum_expr}) / ({count_expr})")
)

df_with_brackets = df_with_avg_rank.withColumn(
    "rank_bracket",
    when(col("avg_rank") >= 80, "Immortal (80+)")
    .when(col("avg_rank") >= 70, "Divine (70-79)")
    .when(col("avg_rank") >= 60, "Ancient (60-69)")
    .when(col("avg_rank") >= 50, "Legend (50-59)")
    .when(col("avg_rank") >= 40, "Archon (40-49)")
    .when(col("avg_rank") >= 30, "Crusader (30-39)")
    .when(col("avg_rank") >= 20, "Guardian (20-29)")
    .when(col("avg_rank") >= 10, "Herald (10-19)")
    .when(col("avg_rank") > 0, "Unranked/Low (<10)")
    .otherwise("No Rank Data")
)

print(f"Total matches processed: {df_with_brackets.count()}")
print(f"Matches with rank data: {df_with_brackets.filter(col('avg_rank') > 0).count()}")

Total matches processed: 1275
Matches with rank data: 1275


In [0]:
from pyspark.sql.functions import count

rank_distribution = df_with_brackets.groupBy("rank_bracket").agg(
    count("*").alias("match_count")
).orderBy(col("match_count").desc())

print("=== RANK DISTRIBUTION ===")
display(rank_distribution)

=== RANK DISTRIBUTION ===


rank_bracket,match_count
Archon (40-49),330
Crusader (30-39),276
Legend (50-59),254
Guardian (20-29),174
Ancient (60-69),114
Herald (10-19),66
Divine (70-79),42
Immortal (80+),19


In [0]:
# Filter for Divine+ (rank 70+)
divine_plus = df_with_brackets.filter(col("avg_rank") >= 70).orderBy(col("avg_rank").desc())

print(f"Total Divine+ matches: {divine_plus.count()}")
print()

display(divine_plus.select(
    "match_id",
    "collection_date", 
    "avg_rank",
    "rank_bracket",
    "duration",
    "radiant_win",
    "game_mode"
))

=== DIVINE+ MATCHES (Rank 70+) ===
Total Divine+ matches: 61



match_id,collection_date,avg_rank,rank_bracket,duration,radiant_win,game_mode
8958422709,2026-08-22,80.0,Immortal (80+),971,false,23
8958622005,2026-08-22,80.0,Immortal (80+),931,true,22
8958811187,2026-08-22,80.0,Immortal (80+),771,false,1
8958357357,2026-08-21,80.0,Immortal (80+),424,true,22
8958813558,2026-08-22,80.0,Immortal (80+),803,true,1
8958935026,2026-08-22,80.0,Immortal (80+),587,true,22
8957885230,2026-08-21,80.0,Immortal (80+),760,true,1
8957884547,2026-08-21,80.0,Immortal (80+),443,true,21
8958271624,2026-08-21,80.0,Immortal (80+),902,true,22
8958025769,2026-08-21,80.0,Immortal (80+),673,false,22


In [0]:
from pyspark.sql.functions import round as spark_round, avg, sum as spark_sum

total_matches = df_with_brackets.filter(col("avg_rank") > 0).count()

bracket_summary = df_with_brackets.filter(col("avg_rank") > 0).groupBy("rank_bracket").agg(
    count("*").alias("matches"),
    spark_round(avg("duration"), 0).alias("avg_duration_sec"),
    spark_round(avg("avg_rank"), 1).alias("avg_rank_tier"),
    spark_round((count("*") / total_matches * 100), 1).alias("pct_of_total"),
    spark_sum(when(col("radiant_win"), 1).otherwise(0)).alias("radiant_wins"),
    spark_sum(when(~col("radiant_win"), 1).otherwise(0)).alias("dire_wins")
).orderBy(col("avg_rank_tier").desc())

bracket_summary = bracket_summary.withColumn(
    "radiant_win_rate",
    spark_round((col("radiant_wins") / col("matches") * 100), 1)
)

print(f"Total matches with rank data: {total_matches}")
print(f"Matches without rank data: {dota2_detailed_matches.count() - total_matches}")
print()

display(bracket_summary.select(
    "rank_bracket",
    "matches",
    "pct_of_total",
    "avg_rank_tier",
    "avg_duration_sec",
    "radiant_win_rate",
    "radiant_wins",
    "dire_wins"
))

=== COMPLETE RANK BRACKET ANALYSIS ===
Total matches with rank data: 1275
Matches without rank data: 0



rank_bracket,matches,pct_of_total,avg_rank_tier,avg_duration_sec,radiant_win_rate,radiant_wins,dire_wins
Immortal (80+),19,1.5,80.0,596.0,68.4,13,6
Divine (70-79),42,3.3,73.7,650.0,45.2,19,23
Ancient (60-69),114,8.9,63.9,693.0,48.2,55,59
Legend (50-59),254,19.9,53.8,630.0,42.9,109,145
Archon (40-49),330,25.9,44.3,671.0,50.0,165,165
Crusader (30-39),276,21.6,34.9,738.0,60.1,166,110
Guardian (20-29),174,13.6,24.5,732.0,46.6,81,93
Herald (10-19),66,5.2,15.0,811.0,56.1,37,29


In [0]:
df_no_turbo = df_with_brackets.filter(col("game_mode") != 23)

print(f"Total matches: {df_with_brackets.count()}")
print(f"Turbo matches: {df_with_brackets.filter(col('game_mode') == 23).count()}")
print(f"Non-Turbo matches: {df_no_turbo.count()}")
print()

total_no_turbo = df_no_turbo.filter(col("avg_rank") > 0).count()

bracket_summary_no_turbo = df_no_turbo.filter(col("avg_rank") > 0).groupBy("rank_bracket").agg(
    count("*").alias("matches"),
    spark_round(avg("duration"), 0).alias("avg_duration_sec"),
    spark_round(avg("avg_rank"), 1).alias("avg_rank_tier"),
    spark_round((count("*") / total_no_turbo * 100), 1).alias("pct_of_total"),
    spark_sum(when(col("radiant_win"), 1).otherwise(0)).alias("radiant_wins"),
    spark_sum(when(~col("radiant_win"), 1).otherwise(0)).alias("dire_wins")
).orderBy(col("avg_rank_tier").desc())

bracket_summary_no_turbo = bracket_summary_no_turbo.withColumn(
    "radiant_win_rate",
    spark_round((col("radiant_wins") / col("matches") * 100), 1)
).withColumn(
    "avg_duration_min",
    spark_round(col("avg_duration_sec") / 60, 1)
)

display(bracket_summary_no_turbo.select(
    "rank_bracket",
    "matches",
    "pct_of_total",
    "avg_rank_tier",
    "avg_duration_min",
    "avg_duration_sec",
    "radiant_win_rate"
))

Total matches: 1275
Turbo matches: 846
Non-Turbo matches: 429



rank_bracket,matches,pct_of_total,avg_rank_tier,avg_duration_min,avg_duration_sec,radiant_win_rate
Immortal (80+),16,3.7,80.0,8.9,534.0,68.8
Divine (70-79),23,5.4,73.8,7.9,475.0,34.8
Ancient (60-69),47,11.0,63.7,8.8,526.0,40.4
Legend (50-59),87,20.3,53.5,7.0,421.0,29.9
Archon (40-49),110,25.6,44.0,8.7,521.0,35.5
Crusader (30-39),73,17.0,34.8,9.6,578.0,43.8
Guardian (20-29),61,14.2,23.8,6.5,387.0,19.7
Herald (10-19),12,2.8,14.7,6.1,366.0,25.0
